In [1]:
import pandas as pd
import numpy as np
import random
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

# Reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported.")

Libraries imported.


In [2]:
# Task 1: Load data from train.csv into a pandas DataFrame
df = pd.read_csv("train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# Task 2: Show number of rows and columns
print(f"Rows:    {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

# Bonus — see column names and dtypes
print("\nDataset info:")
df.info()

Rows:    891
Columns: 12

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
# Task 3
df_opt1 = df.dropna()
print(f"Option 1 (dropna):  {df_opt1.shape[0]} rows left "
      f"(lost {df.shape[0] - df_opt1.shape[0]} rows)")

df_opt2 = df.drop(columns=["Cabin"])
print(f"Option 2 (drop Cabin column):  {df_opt2.shape[1]} columns left")

df_opt3 = df.copy()
df_opt3["Age"] = df_opt3["Age"].fillna(df_opt3["Age"].mean())
df_opt3["Embarked"] = df_opt3["Embarked"].fillna(df_opt3["Embarked"].mode()[0])
print(f"Option 3 (impute Age + Embarked):  missing now = "
      f"{df_opt3.isnull().sum().sum() - df_opt3['Cabin'].isnull().sum()}")

Option 1 (dropna):  183 rows left (lost 708 rows)
Option 2 (drop Cabin column):  11 columns left
Option 3 (impute Age + Embarked):  missing now = 0


**Which columns were chosen for each option, and why?**

For **Option 1 (`dropna`)**, this method is only appropriate for a relatively clean version of the dataset, not for the raw Titanic data. The raw dataset contains too many missing values in `Cabin`, with 687 out of 891 entries missing, or about 77.1%, so applying `dropna()` directly would remove most of the rows. A more practical approach is to drop `Cabin` first and only then apply `dropna()` to the remaining columns.

For **Option 2 (dropping columns)**, `Cabin` was selected because it has by far the largest amount of missing data: 687 missing values out of 891, which is about 77.1%. With so little observed information, imputing values would mostly create artificial data rather than preserve real signal, so removing the column is the safer choice.

For **Option 3 (`fillna`)**, `Age` was selected because it is a numeric feature with 177 missing values, or about 19.9%, which makes mean imputation a reasonable baseline strategy. `Embarked` was handled differently because it is a categorical feature with only 2 missing values, so filling it with the mode, the most frequent category, is more appropriate than using a mean.

The combined strategy used below is therefore: **drop `Cabin`, impute `Age` with the mean, and impute `Embarked` with the mode**.

In [5]:
# Task 4: Use nunique() to find columns unsuitable for modeling
print("Unique values per column:")
print(df.nunique())
print()

df_clean = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Now handle missing values on what's left (combined strategy from Task 3)
df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].mean())
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])

print(f"After cleaning: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print(f"Remaining missing values: {df_clean.isnull().sum().sum()}")
df_clean.head()

Unique values per column:
PassengerId    891
Survived         2
Pclass           3
Name           891
Sex              2
Age             88
SibSp            7
Parch            7
Ticket         681
Fare           248
Cabin          147
Embarked         3
dtype: int64

After cleaning: 891 rows × 8 columns
Remaining missing values: 0


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [6]:
# Task 5: Convert categorical variables with LabelEncoder
# 'Sex' and 'Embarked' are categorical (strings) — models need numbers.

le_sex = LabelEncoder()
df_clean["Sex"] = le_sex.fit_transform(df_clean["Sex"])
# Now: female=0, male=1 (or similar — depends on alphabetical order)

le_emb = LabelEncoder()
df_clean["Embarked"] = le_emb.fit_transform(df_clean["Embarked"])
# Now: C=0, Q=1, S=2

print("Sex mapping:     ", dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_))))
print("Embarked mapping:", dict(zip(le_emb.classes_, le_emb.transform(le_emb.classes_))))

df_clean.head()

Sex mapping:      {'female': np.int64(0), 'male': np.int64(1)}
Embarked mapping: {'C': np.int64(0), 'Q': np.int64(1), 'S': np.int64(2)}


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2


In [7]:
# Task 6: MinMaxScaler — bring all numeric features to the [0, 1] range

cols_to_scale = ["Age", "Fare", "SibSp", "Parch", "Pclass", "Sex", "Embarked"]

scaler = MinMaxScaler()
df_clean[cols_to_scale] = scaler.fit_transform(df_clean[cols_to_scale])

print("After scaling — every feature should have min=0 and max=1:")
df_clean.describe().loc[["min", "max"]]

After scaling — every feature should have min=0 and max=1:


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [9]:
# Task 7: Split into train (80%) and test (20%)

X = df_clean.drop(columns=["Survived"])   
y = df_clean["Survived"]                  

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       
    random_state=42,     
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

X_train: (712, 7)
X_test:  (179, 7)
y_train: (712,)
y_test:  (179,)


In [11]:
# Task 8: Random classifier that returns 0 or 1

def classify(x):
    return random.randint(0, 1)


def run(f_classify, x):
    return list(map(f_classify, x))


def evaluate(predictions, actual):
    correct = list(filter(
        lambda item: item[0] == item[1],
        list(zip(predictions, actual))
    ))
    return f"{len(correct)} correct answers from {len(actual)}. Accuracy ({len(correct)/len(actual)*100:.0f}%)"


# Evaluate the random classifier on the training set
print("Random classifier on TRAIN set:")
print(evaluate(run(classify, X_train.values), y_train.values))

# And on the test set
print("\nRandom classifier on TEST set:")
print(evaluate(run(classify, X_test.values), y_test.values))

Random classifier on TRAIN set:
339 correct answers from 712. Accuracy (48%)

Random classifier on TEST set:
99 correct answers from 179. Accuracy (55%)
